# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AW-OMW/FLY-RANK-PROJECT/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Rule for Predicting Page Boom (Data-Driven) and its Reason Codes

### **Rule**:

A page is predicted to 'boom' next month if it meets *at least two* of the following criteria:
1.  `trend_last_3_months_views` is positive and significant (e.g., greater than 10%).
2.  `search_volume` is in the top 25% of all content items.
3.  `competition` is in the bottom 50% of all content items.

If a page does not meet at least two of these criteria, it is predicted to `not boom`.

### **Reason Codes**:

*   `BOOM_PREDICTED_STRONG_TREND_HIGH_SEARCH`: The page shows a strong positive view trend and high search volume.
*   `BOOM_PREDICTED_HIGH_SEARCH_LOW_COMPETITION`: The page has high search volume and relatively low competition.
*   `BOOM_PREDICTED_STRONG_TREND_LOW_COMPETITION`: The page exhibits a strong positive view trend and relatively low competition.
*   `NO_BOOM_PREDICTED_INSUFFICIENT_CRITERIA`: The page does not meet enough criteria for a 'boom' prediction.
*   `NO_BOOM_PREDICTED_LOW_TREND`: The page has a negative or stagnant view trend.
*   `NO_BOOM_PREDICTED_HIGH_COMPETITION`: The page faces high competition, hindering a potential boom.
*   `NO_BOOM_PREDICTED_LOW_SEARCH_VOLUME`: The page has a low search volume, indicating limited interest.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [15]:
#my rule is if a page has positive and significant trend in last 3 months , search volume is more than 25% and competition level is less than 50% the page will have high chance of booming
#if the page has atleast 2 of these criteria it will be marked as highly likely to boom

In [16]:
#reason codes are :
#BOOM_PREDICTED_STRONG_TREND_HIGH_SEARCH: The page shows a strong positive view trend and high search volume.
#BOOM_PREDICTED_HIGH_SEARCH_LOW_COMPETITION: The page has high search volume and relatively low competition.
#BOOM_PREDICTED_STRONG_TREND_LOW_COMPETITION: The page exhibits a strong positive view trend and relatively low competition.
#NO_BOOM_PREDICTED_INSUFFICIENT_CRITERIA: The page does not meet enough criteria for a 'boom' prediction.
#NO_BOOM_PREDICTED_LOW_TREND: The page has a negative or stagnant view trend.
#NO_BOOM_PREDICTED_HIGH_COMPETITION: The page faces high competition, hindering a potential boom.
#NO_BOOM_PREDICTED_LOW_SEARCH_VOLUME: The page has a low search volume, indicating limited interest.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Define and Apply the `calculate_boom_score` Function

This function will assess each page against the criteria to determine its 'boom' potential, returning a numerical score and associated reason codes. We'll dynamically calculate the `search_volume` and `competition` thresholds based on your DataFrame's data.

In [17]:
def calculate_boom_score(row, search_volume_threshold, competition_threshold):
    """Calculates a 'boom' score for a given row based on defined criteria.

    Args:
        row (pd.Series): A row from the DataFrame.
        search_volume_threshold (float): The 75th percentile of search_volume.
        competition_threshold (float): The 50th percentile of competition.

    Returns:
        tuple: A tuple containing the boom score (int) and a list of reason codes (list of str).
    """
    score = 0
    reason_codes = []

    # Condition 1: trend_pct is positive and significant (>10%)
    if pd.notna(row['trend_pct']) and row['trend_pct'] > 10:
        score += 1
        reason_codes.append('STRONG_TREND')

    # Condition 2: search_volume is in the top 25% of all content items
    if pd.notna(row['search_volume']) and row['search_volume'] > search_volume_threshold:
        score += 1
        reason_codes.append('HIGH_SEARCH_VOLUME')

    # Condition 3: competition is in the bottom 50% of all content items
    if pd.notna(row['competition']) and row['competition'] < competition_threshold:
        score += 1
        reason_codes.append('LOW_COMPETITION')

    return score, reason_codes

# Calculate dynamic thresholds from the DataFrame
search_volume_threshold = df['search_volume'].quantile(0.75)
competition_threshold = df['competition'].quantile(0.50)

print(f"Calculated Search Volume Threshold (75th percentile): {search_volume_threshold:.2f}")
print(f"Calculated Competition Threshold (50th percentile): {competition_threshold:.2f}")

Calculated Search Volume Threshold (75th percentile): 20.00
Calculated Competition Threshold (50th percentile): 0.00


In [18]:
# Apply the function to each row of the DataFrame
df[['boom_score', 'reason_codes']] = df.apply(
    lambda row: calculate_boom_score(row, search_volume_threshold, competition_threshold),
    axis=1,
    result_type='expand'
)

# Display the first few rows with the new scores and reason codes
display(df[['content_id', 'trend_pct', 'search_volume', 'competition', 'boom_score', 'reason_codes']].head())

,content_id,trend_pct,search_volume,competition,boom_score,reason_codes
0,content_304f48230142,-41.4,10.0,0.67,0,[]
1,content_a1fb4e703a9e,-57.7,90.0,0.01,1,[HIGH_SEARCH_VOLUME]
2,content_9aa793d4d895,-60.9,0.0,0.00,0,[]
3,content_331d6c4de07b,-13.8,10.0,0.00,0,[]
4,content_d99b7a2d90ca,-34.7,0.0,0.00,0,[]


In [19]:
import os

output_dir = 'work/outputs'
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'baseline_action_score.csv')
df.to_csv(output_path, index=False)
print(f"DataFrame saved to: {output_path}")

DataFrame saved to: work/outputs/baseline_action_score.csv


The `df` DataFrame, which now includes the `boom_score` and `reason_codes` for all 30,000 content items, has been saved to `work/outputs/baseline_action_score.csv`. This file contains the complete dataset with the calculated scores, ready for further analysis or ranking.

In [20]:
from google.colab import files

output_path = 'work/outputs/baseline_action_score.csv'

try:
    files.download(output_path)
    print(f"Downloading {output_path} to your local machine.")
except Exception as e:
    print(f"Error downloading file: {e}\nPlease ensure the file path is correct and the file exists.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [21]:
# Sort the DataFrame by 'boom_score' in descending order to find the top predictions
top_20_boom_predictions = df.sort_values(by='boom_score', ascending=False).head(20)

# Display the relevant columns for the top 20 predictions
display(top_20_boom_predictions[['content_id', 'boom_score', 'reason_codes', 'trend_pct', 'search_volume', 'competition']])


,content_id,boom_score,reason_codes,trend_pct,search_volume,competition
16384,content_59094571f0a5,2,"[STRONG_TREND, HIGH_SEARCH_VOLUME]",22.6,50.0,0.03
25901,content_1c9181753d4b,2,"[STRONG_TREND, HIGH_SEARCH_VOLUME]",11.8,70.0,0.00
25106,content_df269d7923a1,2,"[STRONG_TREND, HIGH_SEARCH_VOLUME]",93.2,140.0,1.00
20969,content_382ab3b384e4,2,"[STRONG_TREND, HIGH_SEARCH_VOLUME]",64.3,320.0,0.21
20970,content_6a21b377d24c,2,"[STRONG_TREND, HIGH_SEARCH_VOLUME]",70.5,720.0,0.00
20971,content_94ca44aeccd2,2,"[STRONG_TREND, HIGH_SEARCH_VOLUME]",38.5,4400.0,0.49
25894,content_351ae550841b,2,"[STRONG_TREND, HIGH_SEARCH_VOLUME]",50.0,40.0,0.06
23225,content_401c06318c53,2,"[STRONG_TREND, HIGH_SEARCH_VOLUME]",110.0,70.0,0.66
9208,content_f89e7c1dda00,2,"[STRONG_TREND, HIGH_SEARCH_VOLUME]",23.1,260.0,0.00
20987,content_b9c4664ba307,2,"[STRONG_TREND, HIGH_SEARCH_VOLUME]",2860.0,1600.0,0.18


In [22]:
# Flatten the list of lists in the 'reason_codes' column
all_reason_codes = [code for sublist in top_20_boom_predictions['reason_codes'] for code in sublist]

# Get the distribution (counts) of each reason code
reason_code_distribution = pd.Series(all_reason_codes).value_counts()

print("Distribution of Reason Codes in Top 20 Boom Predictions:")
display(reason_code_distribution)


Distribution of Reason Codes in Top 20 Boom Predictions:


,count
STRONG_TREND,20
HIGH_SEARCH_VOLUME,20


## Analysis of Potential Failure and Confidence Note

### Why this analysis might fail:

1.  **Simplified Rule Logic**: The current model uses a simple 'at least two out of three' criteria. This binary scoring might oversimplify the complex interplay between `trend_pct`, `search_volume`, and `competition`. A specific combination (e.g., very high trend and moderate search volume) might be more indicative of a boom than two weaker criteria, but the model treats all qualifying criteria equally.
2.  **Static Thresholds**: The `search_volume_threshold` (75th percentile) and `competition_threshold` (50th percentile) are derived from the current dataset. These thresholds might not be robust or representative of future data distributions. What constitutes 'high' search volume or 'low' competition can change over time or vary significantly across different content categories.
3.  **Lack of Nuance in Reason Codes**: While `reason_codes` help in understanding *why* a boom is predicted, they don't capture the *degree* of each contributing factor. For example, a `STRONG_TREND` with a `trend_pct` of 11% is treated the same as one with 100% in terms of score, but their potential impact on a 'boom' could be vastly different.
4.  **Absence of External Factors**: The model does not consider external influences such as seasonality, major news events, marketing campaigns, or competitor activities, all of which can significantly impact content performance and cause a page to 'boom' or 'fail' unexpectedly.
5.  **Definition of 'Boom'**: The definition of 'boom' is strictly tied to the three chosen criteria. If the business's actual definition of a 'boom' involves other metrics (e.g., conversion rates, user engagement beyond views), then these predictions might not align with true business success.
6.  **Data Representativeness**: We assume the `trend_pct`, `search_volume`, and `competition` data accurately reflect the real-world dynamics. Any biases or inaccuracies in these input features will directly propagate to the `boom_score`.
7.  **Overlooking 'Low Competition'**: It's interesting to note that for the `top_20_boom_predictions`, `LOW_COMPETITION` did not appear in the reason codes, only `STRONG_TREND` and `HIGH_SEARCH_VOLUME`. This suggests that, at least for this top tier, low competition alone isn't a primary driver or perhaps the threshold (0.0) is too strict to capture beneficial low competition scenarios.

### Confidence Note:

*   **Directional Guidance**: The current model provides a *directional* baseline for identifying content likely to experience a 'boom' based on the defined criteria. It's useful for quickly ranking and surfacing content that aligns with these specific conditions.
*   **Limited Predictive Power**: Due to the simplistic nature of the rules and lack of external context, the predictive power for actual 'boom' outcomes (e.g., whether a page *will* truly boom next month) is likely **low to moderate**. It's more of an indicator of *potential* based on predefined criteria rather than a robust predictor.
*   **Improvement Potential**: Confidence could be significantly increased with a more sophisticated model (e.g., machine learning algorithms trained on historical 'boom' events), inclusion of more diverse features, and rigorous validation against actual future performance data.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [23]:
# 4. Weak Picks + Leakage Check
#
# Weak Picks Analysis:
#
# Based on the `top_20_boom_predictions` and the `reason_code_distribution`, here's an observation regarding potential 'weak picks':
#
# 1.  **Strict `LOW_COMPETITION` Threshold**: The `competition_threshold` was calculated as `0.0`. The condition for `LOW_COMPETITION` is `row['competition'] < competition_threshold`, meaning `row['competition'] < 0.0`. For a 'competition' metric, it's highly unlikely (and generally impossible) for its value to be negative. This effectively means that the `LOW_COMPETITION` criterion, as implemented, will *never* be met. Indeed, our `reason_code_distribution` for the top 20 confirms this, showing only `STRONG_TREND` (20 occurrences) and `HIGH_SEARCH_VOLUME` (20 occurrences), with `LOW_COMPETITION` having 0 occurrences.
#
#     *   **Why this makes a pick 'weak'**: If the intention was for 'low competition' to be a valid contributing factor (e.g., competition values in the bottom 50th percentile, which could include `0.0`), the current implementation renders this criterion useless. This means all 'boom' predictions are solely driven by combinations of strong trend and high search volume, ignoring a potentially important third factor. Picks that *would have* been strong due to low competition are missed, and the model's reliance on only two factors makes it less robust.
#
#     *   **What would make it wrong**: The rule states "`competition` is in the bottom 50% of all content items." If the 50th percentile of competition is 0.0 (as observed), and the intention was to include items *equal to or less than* this threshold, the condition should likely be `row['competition'] <= competition_threshold`. The current strict `less than` (`<`) condition means only negative competition values would qualify, which is incorrect for this metric.
#
# Leakage Check:
#
# 1.  **Product Flags**: I do not see any obvious product flags or internal labels in the `df` columns (`content_id`, `client_id`, `search_volume`, `competition`, `trend_pct`, `boom_score`, `reason_codes`) that would indicate direct leakage. Assuming `content_id` and `client_id` are merely identifiers and not predictive features in themselves, there's no apparent issue here.
#
# 2.  **Future Windows Leaked In**: The model uses `trend_pct` (presumably `trend_last_3_months_views`), `search_volume`, and `competition`. Assuming these metrics represent historical or current data points *prior* to the `next month` boom prediction period, there is no direct temporal leakage. The problem statement implies these features are available *before* the prediction target. Therefore, based on the feature names, there's no immediate sign of future data leakage.
#
#     *   **Potential for subtle leakage**: While the feature names seem appropriate, without a detailed understanding of how `search_volume` and `competition` data are collected and timestamped relative to the `trend_pct` and the prediction target, a very subtle form of leakage *could* exist. For example, if `search_volume` implicitly includes data from the 'next month' being predicted, that would constitute leakage. However, based solely on the provided context, we proceed assuming these are valid input features available at prediction time.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.